In [ ]:
from pathlib import Path
import time

import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter
import numpy as np
import pandas as pd

In [ ]:
PARQ_PATH = Path('../sample_dataset/processed_data/parquets/20220730-0001.parquet')
DC_OFFSET = 1.9  # Volts

In [ ]:
# Load the parquet file into a DataFrame
start = time.time()
df = pd.read_parquet(PARQ_PATH)
# parquet stores column names as strings; we want integers
df.columns = df.columns.astype('int16')
load_parquet_time = time.time() - start

num_initial_signals = df.shape[0]

print(
    f"\x1b[1;36m{num_initial_signals}\x1b[0m signals loaded in "
    f"\x1b[1;32m{(load_parquet_time)*1000:.2f} ms\x1b[0m."
)

In [ ]:
# Drop any signals that have missing or infinite values
start = time.time()
df = df.replace([np.inf, -np.inf], np.nan).dropna(how='any')
remove_missing_time = time.time() - start
num_missing_signals = num_initial_signals - df.shape[0]

print(
    f"Removed \x1b[1;31m{num_missing_signals}\x1b[0m signals "
    f"with missing or infinite values. "
    f"[\x1b[1;32m{(remove_missing_time)*1000:.2f} ms\x1b[0m]"
)

In [ ]:
# Subtract DC offset
df = df - DC_OFFSET
# Invert signals
df = -1 * df

In [ ]:
# Offset dataframe upwards by global minimum
# TODO: check if this is necessary
df = df + abs(min(df.min()))

In [ ]:
# Plot signals
fig, ax = plt.subplots(figsize=(15,8))

ax.plot(
    df.T
)

ax.grid(visible=True)

ax.tick_params(axis='both', labelsize=14)
ax.set_xlabel("time (a.u.)", fontsize=18)
ax.set_ylabel("amplitude ($V$)", fontsize=18)

fig.show()

In [ ]:
# Remove signals with initial values greater than some threshold
start = time.time()
threshold = 0.15  # volts
filt = (df[0] < threshold)
df_clean = df[filt]
remove_nonpeaks_time = time.time() - start

num_filtered_signals = df.shape[0] - df_clean.shape[0]

print(
    f"Removed \x1b[1;31m{num_filtered_signals}\x1b[0m signals "
    f"with starting values greater than \x1b[1;31m{threshold}\x1b[0m volts. "
    f"[\x1b[1;32m{(remove_nonpeaks_time)*1000:.2f} ms\x1b[0m]"    
)

In [ ]:
print(
    f"Total cleaning time: {(load_parquet_time + remove_missing_time + remove_nonpeaks_time)*1000:.2f} ms"
)

In [ ]:
# Plot filtered signals
fig, ax = plt.subplots(figsize=(15,8))

signal_range = [0, 10]

ax.plot(
    df_clean.iloc[signal_range[0]:signal_range[1]].T
)

ax.grid(visible=True)

ax.tick_params(axis='both', labelsize=14)
ax.set_xlabel("time (a.u.)", fontsize=18)
ax.set_ylabel("amplitude ($V$)", fontsize=18)

fig.show()

In [ ]:
df_clean.columns = df_clean.columns.astype(str)
df_clean.to_parquet("../sample_dataset/processed_data/cleaned_parquets/20220730-0001_c.parquet")